# Подключение

### Импорты

In [2]:
import os
import tqdm

import numpy as np
import polars as pl
from io import BytesIO
from datasets import load_dataset

In [3]:
import boto3
from botocore.client import Config

In [ ]:
YOUR_KEY_ID = None # id статического ключа
YOUR_SECRET_KEY = None # секрет статического ключа

In [ ]:
BUCKET_NAME = None # название бакета (без s3://)
ENDPOINT_URL = None  # Стандартный эндпоинт
REGION = None 

### Скачиваем

#### Создаем клиент для S3

In [94]:
session = boto3.session.Session()
s3_client = session.client(
    's3',
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=YOUR_KEY_ID,
    aws_secret_access_key=YOUR_SECRET_KEY,
    config=Config(region_name=REGION) 
)

Создаем бакет

In [19]:

bucket_name = BUCKET_NAME

try:
    s3_client.create_bucket(Bucket=bucket_name)
    print(f"Бакет '{bucket_name}' успешно создан!")
except Exception as e:
    print(f"Не удалось создать бакет: {e}")

Бакет 'kateiner' успешно создан!


#### Debug

In [85]:
# Проверяем бакет

from botocore.exceptions import ClientError


try:
    # Отправляем HEAD-запрос к бакету
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Бакет '{BUCKET_NAME}' существует, и у вас есть к нему доступ.")
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code == '404':
        # Бакет не найден
        print(f"❌ Бакет '{BUCKET_NAME}' не существует.")
    elif error_code == '403':
        # Бакет есть, но нет прав (или он принадлежит другому аккаунту)
        print(f"⛔ Доступ к бакету '{BUCKET_NAME}' запрещён (403 Forbidden).")
    else:
        # Другие ошибки (например, проблемы с сетью или неверный endpoint)
        print(f"⚠️ Ошибка при проверке бакета: {e}")

✅ Бакет 'kateiner' существует, и у вас есть к нему доступ.


In [86]:
# Узнаем список бакетоа

# Используем s3_client, который вы создали ранее
response = s3_client.list_buckets()

# Проверяем, есть ли бакеты в ответе
if 'Buckets' in response:
    print("Список доступных бакетов:")
    for bucket in response['Buckets']:
        # Выводим имя и дату создания каждого бакета
        print(f" - {bucket['Name']} (создан: {bucket['CreationDate']})")
else:
    print("Бакеты не найдены или у вас нет прав на их просмотр.")

Список доступных бакетов:
 - kateiner (создан: 2026-10-24 10:24:47.317000+00:00)


# Собираем данные

#### Likes

##### Загрузка

In [ ]:
likes = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet") # Загружаем в оперативку

In [7]:
likes.keys()

dict_keys(['train'])

In [ ]:
likes = likes['train'].to_polars()

In [10]:
likes.select(pl.col("uid").n_unique())

uid
u32
8283


In [11]:
likes.head()

uid,timestamp,item_id,is_organic
u32,u32,u32,u8
100,44755,732449,1
100,1155860,6568592,0
100,1259125,5411243,1
100,1260005,7371186,0
100,1263935,4943655,0


In [47]:
likes_train.select(pl.col("timestamp").max())

timestamp
u32
25999970


In [46]:
likes_train.select(pl.col("timestamp").min())

timestamp
u32
80


##### Подключаем родные модули YAMBDA

In [13]:
import subprocess
import sys
import os

In [16]:
from huggingface_hub import snapshot_download

In [ ]:
repo_dir = "yambda_scripts"

# Скачать только все файлы из папки benchmarks/yambda
snapshot_download(
    repo_id="yandex/yambda",
    repo_type="dataset",
    allow_patterns="benchmarks/yambda/*",  # Ключевой момент: указываем скачивание только файлов из этой папки
    local_dir="./yambda_scripts"       # Укажите локальную папку для сохранения
)


In [76]:
import os, sys
print("cwd:", os.getcwd())
print("__file__ ноутбука:", globals().get("__file__", "нет __file__ (это ноутбук)"))
print("\nsys.path:")
for p in sys.path:
    print("  ", p)
print("\nСуществует ли ./benchmarks/yambda/processing/timesplit.py из cwd?",
      os.path.exists(os.path.join(os.getcwd(), "benchmarks", "yambda", "processing", "timesplit.py")))

cwd: /home/jupyter/project/lab
__file__ ноутбука: нет __file__ (это ноутбук)

sys.path:
   /home/jupyter/project/lab
   
   /kernel/numpy-fallback/lib/python3.10/site-packages
   /usr/lib/python310.zip
   /usr/lib/python3.10
   /usr/lib/python3.10/lib-dynload
   /kernel/lib/python3.10/site-packages
   /home/jupyter/.local/lib/python3.10/site-packages
   /usr/local/lib/python3.10/dist-packages
   /usr/lib/python3/dist-packages
   /kernel/fallback/lib/python3.10/site-packages
   /kernel/lib/python3.10/site-packages/ml_kernel/_vendor/IPython/extensions
   /kernel/lib/python3.10/site-packages/setuptools/_vendor
   /home/jupyter/.ipython
   /kernel/lib/python3.10/site-packages/ml_kernel/_vendor/IPython/extensions

Существует ли ./benchmarks/yambda/processing/timesplit.py из cwd? True


In [77]:
import os, sys, importlib.util

# 1. Чистим sys.path от дублей и мусорных путей, связанных с benchmarks,
#    чтобы не было конфликтов с разными копиями пакета
sys.path = [p for p in sys.path if "benchmarks" not in p]

# 2. Корень пакета — там, где лежит папка yambda (это .../lab/benchmarks)
yambda_root = os.path.abspath("./benchmarks")
if not os.path.isdir(os.path.join(yambda_root, "yambda")):
    raise FileNotFoundError(f"Не найдена папка yambda в {yambda_root}")

# 3. Регистрируем пакеты yambda и yambda.processing как настоящие пакеты,
#    чтобы относительные импорты внутри timesplit.py работали
import types

pkg_yambda = types.ModuleType("yambda")
pkg_yambda.__path__ = [os.path.join(yambda_root, "yambda")]
sys.modules["yambda"] = pkg_yambda

pkg_proc = types.ModuleType("yambda.processing")
pkg_proc.__path__ = [os.path.join(yambda_root, "yambda", "processing")]
sys.modules["yambda.processing"] = pkg_proc

# 4. Загружаем timesplit.py под полным именем yambda.processing.timesplit
file_path = os.path.join(yambda_root, "yambda", "processing", "timesplit.py")
spec = importlib.util.spec_from_file_location("yambda.processing.timesplit", file_path)
mod = importlib.util.module_from_spec(spec)
sys.modules["yambda.processing.timesplit"] = mod
spec.loader.exec_module(mod)

# 5. Используем
from yambda.processing import timesplit
print("Успех! timesplit:", timesplit)
print("Доступные функции:", [n for n in dir(timesplit) if not n.startswith("_")])

Успех! timesplit: <module 'yambda.processing.timesplit' from '/home/jupyter/project/lab/benchmarks/yambda/processing/timesplit.py'>
Доступные функции: ['Constants', 'flat_split_train_val_test', 'pl', 'sequential_split_train_val_test']


In [ ]:
from yambda.processing import timesplit

##### Делим likes

In [82]:
from yambda.constants import Constants
from yambda.processing.timesplit import flat_split_train_val_test

In [ ]:
likes_train, likes_val, likes_test = flat_split_train_val_test(
    df=likes.lazy(),
    test_timestamp=Constants.TEST_TIMESTAMP,
    val_size=Constants.VAL_SIZE,
    gap_size=Constants.GAP_SIZE,
)

In [90]:
likes_train

In [92]:
# 1. Создаём временный локальный путь
import tempfile
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path = tmp.name

try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    likes_train.sink_parquet(local_path)
    key = "data/yambda/likes_train.parquet"
    s3_client.upload_file(local_path, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path):
        os.remove(local_path)

Загружено в s3://{'Name': 'kateiner', 'CreationDate': datetime.datetime(2026, 10, 24, 10, 24, 47, 317000, tzinfo=tzlocal())}/data/yambda/likes_train.parquet


In [107]:
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path = tmp.name

try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    likes_test.sink_parquet(local_path)
    key = "data/yambda/likes_test.parquet"
    s3_client.upload_file(local_path, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path):
        os.remove(local_path)

Загружено в s3://kateiner/data/yambda/likes_test.parquet


In [108]:
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path = tmp.name

try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    likes_val.sink_parquet(local_path)
    key = "data/yambda/likes_val.parquet"
    s3_client.upload_file(local_path, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path):
        os.remove(local_path)

Загружено в s3://kateiner/data/yambda/likes_val.parquet


Проверим, что файл на месте

In [95]:

bucket = BUCKET_NAME
key = "data/yambda/likes_train.parquet"

response = s3_client.head_object(Bucket=bucket, Key=key)
size_mb = response["ContentLength"] / (1024 * 1024)
print(f"Файл существует, размер: {size_mb:.2f} MB")
print("Last-Modified:", response["LastModified"])

Файл существует, размер: 6.55 MB
Last-Modified: 2026-10-25 08:52:46+00:00


In [104]:


s3_path = f"s3://{bucket}/{key}"

# Ленивое чтение — не загружает данные в память
lf = pl.scan_parquet(
    s3_path,
    storage_options={
        "aws_access_key_id": YOUR_KEY_ID,
        "aws_secret_access_key": YOUR_SECRET_KEY,
        "aws_region": 'ru-central1-a',
        "endpoint_url": ENDPOINT_URL
    }
)

# # Схема должна совпадать с исходным likes_train
# print("Схема прочитанного файла:")
# print(lf_check.collect_schema())

# # Количество строк (выполнит запрос, но только count — это дёшево)
# n_rows = lf_check.select(pl.len()).collect().item()
# print(f"Строк в файле: {n_rows}")

In [105]:
n_loaded = lf.select(pl.len()).collect().item()
n_original = likes_train.select(pl.len()).collect().item()

In [106]:
print(f"Строк в S3: {n_loaded}, в оригинале: {n_original}")
assert n_loaded == n_original, "Количество строк не совпадает!"
print("✅ Данные загружены и читаются корректно")

Строк в S3: 872573, в оригинале: 872573
✅ Данные загружены и читаются корректно


#### Dislikes

In [112]:
dislikes = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="dislikes.parquet") 

Generating train split: 107776 examples [00:00, 1247654.83 examples/s]


In [115]:
dislikes = dislikes['train'].to_polars()
dislikes_train, dislikes_val, dislikes_test = flat_split_train_val_test(
    df=dislikes.lazy(),
    test_timestamp=Constants.TEST_TIMESTAMP,
    val_size=Constants.VAL_SIZE,
    gap_size=Constants.GAP_SIZE,
)

In [117]:
name = "dislikes"

with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_train = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_test = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_val = tmp.name

try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    dislikes_test.sink_parquet(local_path_test)
    key = f"data/yambda/{name}_test.parquet"
    s3_client.upload_file(local_path_test, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    dislikes_train.sink_parquet(local_path_train)
    key = f"data/yambda/{name}_train.parquet"
    s3_client.upload_file(local_path_train, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    dislikes_val.sink_parquet(local_path_val)
    key = f"data/yambda/{name}_val.parquet"
    s3_client.upload_file(local_path_val, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path_train):
        os.remove(local_path_train)
    if os.path.exists(local_path_test):
        os.remove(local_path_test)
    if os.path.exists(local_path_val):
        os.remove(local_path_val)

Загружено в s3://kateiner/data/yambda/dislikes_test.parquet
Загружено в s3://kateiner/data/yambda/dislikes_train.parquet
Загружено в s3://kateiner/data/yambda/dislikes_val.parquet


#### Listens

In [113]:
listens = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="listens.parquet") 

Generating train split: 46467212 examples [00:21, 2163178.79 examples/s]


In [116]:
listens = listens['train'].to_polars()
listens_train, listens_val, listens_test = flat_split_train_val_test(
    df=listens.lazy(),
    test_timestamp=Constants.TEST_TIMESTAMP,
    val_size=Constants.VAL_SIZE,
    gap_size=Constants.GAP_SIZE,
)

In [118]:
name = "listens"

with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_train = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_test = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_val = tmp.name

try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    listens_test.sink_parquet(local_path_test)
    key = f"data/yambda/{name}_test.parquet"
    s3_client.upload_file(local_path_test, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    listens_train.sink_parquet(local_path_train)
    key = f"data/yambda/{name}_train.parquet"
    s3_client.upload_file(local_path_train, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    listens_val.sink_parquet(local_path_val)
    key = f"data/yambda/{name}_val.parquet"
    s3_client.upload_file(local_path_val, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path_train):
        os.remove(local_path_train)
    if os.path.exists(local_path_test):
        os.remove(local_path_test)
    if os.path.exists(local_path_val):
        os.remove(local_path_val)

Загружено в s3://kateiner/data/yambda/listens_test.parquet
Загружено в s3://kateiner/data/yambda/listens_train.parquet
Загружено в s3://kateiner/data/yambda/listens_val.parquet


#### Отмененные дизлайки

In [135]:
undislikes = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="undislikes.parquet") 

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7407978c-6b73-41df-961d-25d33bda68e9)')' thrown while requesting HEAD https://huggingface.co/datasets/yandex/yambda/resolve/main/README.md
Retrying in 1s [Retry 1/5].
Generating train split: 21033 examples [00:00, 877903.79 examples/s]


In [136]:
undislikes = undislikes['train'].to_polars()
undislikes_train, undislikes_val, undislikes_test = flat_split_train_val_test(
    df=undislikes.lazy(),
    test_timestamp=Constants.TEST_TIMESTAMP,
    val_size=Constants.VAL_SIZE,
    gap_size=Constants.GAP_SIZE,
)

In [137]:
name = "undislikes"

with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_train = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_test = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_val = tmp.name

try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    undislikes_test.sink_parquet(local_path_test)
    key = f"data/yambda/{name}_test.parquet"
    s3_client.upload_file(local_path_test, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    undislikes_train.sink_parquet(local_path_train)
    key = f"data/yambda/{name}_train.parquet"
    s3_client.upload_file(local_path_train, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    undislikes_val.sink_parquet(local_path_val)
    key = f"data/yambda/{name}_val.parquet"
    s3_client.upload_file(local_path_val, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path_train):
        os.remove(local_path_train)
    if os.path.exists(local_path_test):
        os.remove(local_path_test)
    if os.path.exists(local_path_val):
        os.remove(local_path_val)

Загружено в s3://kateiner/data/yambda/undislikes_test.parquet
Загружено в s3://kateiner/data/yambda/undislikes_train.parquet
Загружено в s3://kateiner/data/yambda/undislikes_val.parquet


# Анализируем

#### Train

In [126]:
liked_users = likes_train.select("uid").unique()
liked_pairs = (
    likes_train
    .select("uid", "item_id")
    .unique()
)

listen_pairs = (
    listens_train
    .group_by(["uid", "item_id"])
    .agg(
        pl.col("played_ratio_pct")
        .max()
        .alias("max_played_pct"),

        (pl.col("is_organic") == 0)
        .any()
        .alias("was_recommended"),
    )
)


D: дизлайки

In [127]:
neg_dislike = (
    dislikes_train
    .select("uid", "item_id")
    .unique()
    .join(liked_pairs, on=["uid", "item_id"], how="anti")
    .join(liked_users, on="uid", how="semi")
    .collect()
)

A: рекомендованные, не дослушанные и не лайкнутые

In [129]:
neg_rec = (
    listen_pairs
    .filter(
        (pl.col("was_recommended"))
        & (pl.col("max_played_pct") <= 50)
    )
    .select("uid", "item_id")
    .join(liked_pairs, on=["uid", "item_id"], how="anti")
    .join(liked_users, on="uid", how="semi")
    .collect()
)

C: любые недослушанные (рекомендованные и не рекомендованные)

In [130]:
neg_skip = (
    listen_pairs
    .filter(pl.col("max_played_pct") <= 50)
    .select("uid", "item_id")
    .join(liked_pairs, on=["uid", "item_id"], how="anti")
    .join(liked_users, on="uid", how="semi")
    .collect()
)

In [131]:
def get_stats(df: pl.DataFrame, name: str) -> dict:
    return {
        "category": name,
        "pairs": df.height,
        "users": df["uid"].n_unique(),
        "items": df["item_id"].n_unique(),
        "pairs_per_user": (
            round(df.height / df["uid"].n_unique(), 1)
            if df.height > 0 else 0
        ),
    }

negative_stats = pl.DataFrame([
    get_stats(neg_rec, "A: Recommended skips"),
    get_stats(neg_skip, "C: All skips"),
    get_stats(neg_dislike, "D: Dislikes"),
])

print(negative_stats)

shape: (3, 5)
┌──────────────────────┬─────────┬───────┬────────┬────────────────┐
│ category             ┆ pairs   ┆ users ┆ items  ┆ pairs_per_user │
│ ---                  ┆ ---     ┆ ---   ┆ ---    ┆ ---            │
│ str                  ┆ i64     ┆ i64   ┆ i64    ┆ f64            │
╞══════════════════════╪═════════╪═══════╪════════╪════════════════╡
│ A: Recommended skips ┆ 2271502 ┆ 7296  ┆ 262211 ┆ 311.3          │
│ C: All skips         ┆ 3876461 ┆ 7508  ┆ 541812 ┆ 516.3          │
│ D: Dislikes          ┆ 92483   ┆ 5111  ┆ 49601  ┆ 18.1           │
└──────────────────────┴─────────┴───────┴────────┴────────────────┘


In [133]:
keys = ["uid", "item_id"]

print(
    "A ∩ D:",
    neg_rec.join(neg_dislike, on=keys, how="inner").height
)

print(
    "C ∩ D:",
    neg_skip.join(neg_dislike, on=keys, how="inner").height
)

neg_all = pl.concat([
    neg_skip,
    neg_dislike,
]).unique()

print("Всего уникальных негативов:", neg_all.height)

A ∩ D: 36249
C ∩ D: 43505
Всего уникальных негативов: 3925439


# Расставляем метки

- Если дизлайк произошёл и не был отменён до конца соответствующего периода, он считается действующим.
- Если после дизлайка был undislike, этот дизлайк больше не даёт метку 0.
- Если после отмены пользователь вновь поставил дизлайк, он снова считается действующим.
- Для val и test учитываем отмены из предыдущих периодов, поскольку они могут определять текущее состояние дизлайка.

In [146]:
KEYS = ["uid", "item_id"]
SKIP_THRESHOLD = 50

COLUMNS = [
    "uid",
    "item_id",
    "label",
    "is_rec_skip",
    "is_dislike",
]

def unique_pairs(df):
    return df.select(KEYS).unique()


def make_negative_pairs(
    listens,
    dislikes,
    undislikes_history,
):
    # A: рекомендательные пропуски
    # Учитываем максимальную долю прослушивания
    # и наличие рекомендательного события.
    skips = (
        listens
        .group_by(KEYS)
        .agg(
            pl.col("played_ratio_pct").max().alias("max_played"),
            (pl.col("is_organic") == 0).any().alias("is_rec"),
        )
        .filter(
            pl.col("is_rec")
            & (pl.col("max_played") <= SKIP_THRESHOLD)
        )
        .select(KEYS)
        .with_columns(pl.lit(True).alias("is_rec_skip"))
    )

    # D: последний дизлайк в текущем периоде
    last_dislikes = (
        dislikes
        .group_by(KEYS)
        .agg(
            pl.col("timestamp").max().alias("last_dislike")
        )
    )

    # Последняя отмена, известная на конец периода
    last_undislikes = (
        undislikes_history
        .group_by(KEYS)
        .agg(
            pl.col("timestamp").max().alias("last_undislike")
        )
    )

    # Оставляем только действующие дизлайки
    active_dislikes = (
        last_dislikes
        .join(last_undislikes, on=KEYS, how="left")
        .filter(
            pl.col("last_undislike").is_null()
            | (
                pl.col("last_dislike")
                > pl.col("last_undislike")
            )
        )
        .select(KEYS)
        .with_columns(pl.lit(True).alias("is_dislike"))
    )

    # Объединяем A и D
    return (
        skips
        .join(active_dislikes, on=KEYS, how="full", coalesce=True)
        .with_columns(
            pl.col("is_rec_skip").fill_null(False),
            pl.col("is_dislike").fill_null(False),
        )
    )



def make_split(
    likes,
    listens,
    dislikes,
    undislikes_history,
    train_users,
    excluded_pairs=None,
):
    # Положительные примеры
    positives = (
        unique_pairs(likes)
        .with_columns(
            pl.lit(1).cast(pl.Int8).alias("label"),
            pl.lit(False).alias("is_rec_skip"),
            pl.lit(False).alias("is_dislike"),
        )
    )

    # Отрицательные примеры
    negatives = (
        make_negative_pairs(
            listens,
            dislikes,
            undislikes_history,
        )
        # Приоритет лайкам внутри текущего периода
        .join(unique_pairs(likes), on=KEYS, how="anti")
        .with_columns(
            pl.lit(0).cast(pl.Int8).alias("label")
        )
    )



    result = pl.concat(
        [
            positives.select(COLUMNS),
            negatives.select(COLUMNS),
        ],
        how="vertical",
    )

    # Только пользователи из train
    result = result.join(
        train_users, on="uid", how="semi"
    )

    # Исключаем ранее известные пары
    if excluded_pairs is not None:
        result = result.join(
            excluded_pairs, on=KEYS, how="anti"
        )

    return result.select(
        "uid", "item_id", "label",
        "is_rec_skip", "is_dislike"
    )


In [147]:
# Пользователи обучающего периода
train_users = (
    likes_train
    .select("uid")
    .unique()
)

# История отмен дизлайков на конец каждого периода
undo_train = undislikes_train

undo_val = pl.concat([
    undislikes_train,
    undislikes_val,
])

undo_test = pl.concat([
    undislikes_train,
    undislikes_val,
    undislikes_test,
])

# TRAIN
train = make_split(
    likes=likes_train,
    listens=listens_train,
    dislikes=dislikes_train,
    undislikes_history=undo_train,
    train_users=train_users,
)

# VAL
val = make_split(
    likes=likes_val,
    listens=listens_val,
    dislikes=dislikes_val,
    undislikes_history=undo_val,
    train_users=train_users,
    excluded_pairs=unique_pairs(train),
)

# TEST
test = make_split(
    likes=likes_test,
    listens=listens_test,
    dislikes=dislikes_test,
    undislikes_history=undo_test,
    train_users=train_users,
    excluded_pairs=pl.concat([
        unique_pairs(train),
        unique_pairs(val),
    ]).unique(),
)


In [148]:
test

Что по балансу?

In [149]:
def split_stats(df: pl.LazyFrame, name: str):
    return (
        df.select(
            pl.lit(name).alias("split"),
            pl.len().alias("total"),
            (pl.col("label") == 1).sum().alias("positive"),
            (pl.col("label") == 0).sum().alias("negative"),
            pl.col("uid").n_unique().alias("users"),
            pl.col("item_id").n_unique().alias("items"),
            pl.col("is_rec_skip").sum().alias("rec_skips"),
            pl.col("is_dislike").sum().alias("dislikes"),
        )
        .with_columns(
            (
                100 * pl.col("positive") / pl.col("total")
            ).round(2).alias("positive_pct"),
            (
                pl.col("negative")
                / pl.col("positive")
            ).round(2).alias("neg_pos_ratio"),
        )
    )


balance = pl.concat([
    split_stats(train, "train"),
    split_stats(val, "val"),
    split_stats(test, "test"),
]).collect()

print(balance)


shape: (3, 10)
┌───────┬─────────┬──────────┬──────────┬───┬───────────┬──────────┬──────────────┬───────────────┐
│ split ┆ total   ┆ positive ┆ negative ┆ … ┆ rec_skips ┆ dislikes ┆ positive_pct ┆ neg_pos_ratio │
│ ---   ┆ ---     ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---      ┆ ---          ┆ ---           │
│ str   ┆ u32     ┆ u32      ┆ u32      ┆   ┆ u32       ┆ u32      ┆ f64          ┆ f64           │
╞═══════╪═════════╪══════════╪══════════╪═══╪═══════════╪══════════╪══════════════╪═══════════════╡
│ train ┆ 3161439 ┆ 836152   ┆ 2325287  ┆ … ┆ 2271502   ┆ 89352    ┆ 26.45        ┆ 2.78          │
│ val   ┆ 27123   ┆ 4213     ┆ 22910    ┆ … ┆ 22780     ┆ 198      ┆ 15.53        ┆ 5.44          │
│ test  ┆ 28502   ┆ 3885     ┆ 24617    ┆ … ┆ 24437     ┆ 332      ┆ 13.63        ┆ 6.34          │
└───────┴─────────┴──────────┴──────────┴───┴───────────┴──────────┴──────────────┴───────────────┘


По пользователям:

In [150]:

def user_balance_stats(df, name):
    by_user = (
        df.group_by("uid")
        .agg(
            (pl.col("label") == 1).sum().alias("pos"),
            (pl.col("label") == 0).sum().alias("neg"),
        )
    )

    return by_user.select(
        pl.lit(name).alias("split"),
        pl.len().alias("users"),
        (pl.col("pos") > 0).sum().alias("users_with_pos"),
        (pl.col("neg") > 0).sum().alias("users_with_neg"),
        (
            (pl.col("pos") > 0)
            & (pl.col("neg") > 0)
        ).sum().alias("users_with_both"),
        pl.col("pos").median().alias("median_pos"),
        pl.col("neg").median().alias("median_neg"),
    )


user_balance = pl.concat([
    user_balance_stats(train, "train"),
    user_balance_stats(val, "val"),
    user_balance_stats(test, "test"),
]).collect()

print(user_balance)


shape: (3, 7)
┌───────┬───────┬────────────────┬────────────────┬─────────────────┬────────────┬────────────┐
│ split ┆ users ┆ users_with_pos ┆ users_with_neg ┆ users_with_both ┆ median_pos ┆ median_neg │
│ ---   ┆ ---   ┆ ---            ┆ ---            ┆ ---             ┆ ---        ┆ ---        │
│ str   ┆ u32   ┆ u32            ┆ u32            ┆ u32             ┆ f64        ┆ f64        │
╞═══════╪═══════╪════════════════╪════════════════╪═════════════════╪════════════╪════════════╡
│ train ┆ 8269  ┆ 8269           ┆ 7898           ┆ 7898            ┆ 42.0       ┆ 126.0      │
│ val   ┆ 2805  ┆ 1208           ┆ 2348           ┆ 751             ┆ 0.0        ┆ 3.0        │
│ test  ┆ 2885  ┆ 1246           ┆ 2403           ┆ 764             ┆ 0.0        ┆ 3.0        │
└───────┴───────┴────────────────┴────────────────┴─────────────────┴────────────┴────────────┘


Пересечения?

In [151]:

def count_overlap(a, b):
    return (
        a.select(KEYS).unique()
        .join(
            b.select(KEYS).unique(),
            on=KEYS,
            how="inner",
        )
        .select(pl.len())
        .collect()
        .item()
    )


print("train ∩ val:", count_overlap(train, val))
print("train ∩ test:", count_overlap(train, test))
print("val ∩ test:", count_overlap(val, test))


train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


### Сделаем тест и вал с пользователями, имеющих хотя бы 1 позитив и 1 негатив

In [153]:
def only_users_with_both_classes(df: pl.LazyFrame):
    valid_users = (
        df.group_by("uid")
        .agg(
            (pl.col("label") == 1).any().alias("has_pos"),
            (pl.col("label") == 0).any().alias("has_neg"),
        )
        .filter(
            pl.col("has_pos") & pl.col("has_neg")
        )
        .select("uid")
    )

    return df.join(valid_users, on="uid", how="semi")


val_ranking = only_users_with_both_classes(val)
test_ranking = only_users_with_both_classes(test)

In [154]:

def candidate_stats(df: pl.LazyFrame):
    per_user = (
        df.group_by("uid")
        .agg(
            pl.len().alias("n_candidates"),
            (pl.col("label") == 1).sum().alias("n_pos"),
            (pl.col("label") == 0).sum().alias("n_neg"),
        )
    )

    return per_user.select(
        pl.len().alias("users"),
        *[
            pl.col("n_candidates")
            .quantile(q)
            .alias(f"candidates_p{int(q * 100)}")
            for q in [0.1, 0.25, 0.5, 0.75, 0.9]
        ],
        (pl.col("n_candidates") >= 10)
        .sum()
        .alias("users_with_10plus"),
        (pl.col("n_candidates") >= 50)
        .sum()
        .alias("users_with_50plus"),
    ).collect()


print(candidate_stats(val_ranking))
print(candidate_stats(test_ranking))


shape: (1, 8)
┌───────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ users ┆ candidates ┆ candidates ┆ candidates ┆ candidates ┆ candidates ┆ users_with ┆ users_with │
│ ---   ┆ _p10       ┆ _p25       ┆ _p50       ┆ _p75       ┆ _p90       ┆ _10plus    ┆ _50plus    │
│ u32   ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│       ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ u32        ┆ u32        │
╞═══════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╡
│ 751   ┆ 2.0        ┆ 4.0        ┆ 9.0        ┆ 20.0       ┆ 38.0       ┆ 362        ┆ 51         │
└───────┴────────────┴────────────┴────────────┴────────────┴────────────┴────────────┴────────────┘
shape: (1, 8)
┌───────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ users ┆ candidates ┆ candidates ┆ candidates ┆ candidates ┆ c

In [155]:
# def users_with_min_candidates(df, min_candidates=10):
#     users = (
#         df.group_by("uid")
#         .agg(pl.len().alias("n_candidates"))
#         .filter(pl.col("n_candidates") >= min_candidates)
#         .select("uid")
#     )
#     return df.join(users, on="uid", how="semi")


# val_ranking_10plus = users_with_min_candidates(
#     val_ranking, min_candidates=10
# )

# test_ranking_10plus = users_with_min_candidates(
#     test_ranking, min_candidates=10
# )

In [158]:
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_train = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_test = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_val = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_ranking_test = tmp.name
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_path_ranking_val = tmp.name

In [162]:
try:
    # 2. Записываем LazyFrame в локальный parquet-файл
    #    sink_parquet эффективно обрабатывает LazyFrame и не загружает все данные в память
    test.sink_parquet(local_path_test)
    key = f"data/yambda/test.parquet"
    s3_client.upload_file(local_path_test, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    train.sink_parquet(local_path_train)
    key = f"data/yambda/train.parquet"
    s3_client.upload_file(local_path_train, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    val.sink_parquet(local_path_val)
    key = f"data/yambda/val.parquet"
    s3_client.upload_file(local_path_val, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    test_ranking.sink_parquet(local_path_train)
    key = f"data/yambda/test_ranking.parquet"
    s3_client.upload_file(local_path_ranking_test, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")
    
    val_ranking.sink_parquet(local_path_val)
    key = f"data/yambda/val_ranking.parquet"
    s3_client.upload_file(local_path_ranking_val, BUCKET_NAME, key)
    print(f"Загружено в s3://{bucket}/{key}")

finally:
    # 4. Удаляем временный файл
    if os.path.exists(local_path_train):
        os.remove(local_path_train)
    if os.path.exists(local_path_test):
        os.remove(local_path_test)
    if os.path.exists(local_path_val):
        os.remove(local_path_val)
    if os.path.exists(local_path_ranking_test):
        os.remove(local_path_ranking_test)
    if os.path.exists(local_path_ranking_val):
        os.remove(local_path_ranking_val)

Загружено в s3://kateiner/data/yambda/test.parquet
Загружено в s3://kateiner/data/yambda/train.parquet
Загружено в s3://kateiner/data/yambda/val.parquet
Загружено в s3://kateiner/data/yambda/test_ranking.parquet
Загружено в s3://kateiner/data/yambda/val_ranking.parquet


In [1]:
 ## Прокидываем timestemp